In [3]:
pip install -U transformers huggingface_hub "torch==2.9.1" --quiet

This notebook explains how to generate structured outputs with LLMs. It follows the procedure proposed by Willard & Louf in "Efficient Guided Generation for Large Language Models"(https://arxiv.org/pdf/2307.09702).

# Why structured outputs?
One of the problems with LLMs is that despite its ability to manipulate language, it is difficult to guarantee outputs in a specific format. For example, suppose we want to extract dates from a corpus of text. Ideally, we would like:
1. Consistent format: we want _all_ the extracted dates to follow the same format (e.g "2022-10-01").
2. No verbosity: most LLMs will reply with "Of course, the date of this event was _2022-10-01_. Anything else I can help you with?" or "The event happened on _2022-10-01_. It was of great importance because...". LLMs often embed the information we want inside extra text, leaving us with another parsing problem.

This section illustrates why out-of-the-box LLMs are not well suited to satisfy these constraints.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import random


device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
token = "hf_..."

We will use Google Gemma 1B for this project.

In [7]:
model_id = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map=device,
    dtype=torch.bfloat16,
    token=token,
)

model.eval()


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

We start with two date-related questions and generate three answers for each.

In [8]:

events = [
    'the final of the World Cup 2022',
    'la prise de la Bastille',
]


answers = [
    '18-12-2022',
    '14-07-1789'
]

questions = [f"What's the date of {e}?" for e in events]

number_of_answers = 3

for q, a in zip(questions, answers):
  print(' * Question :', q)
  print(' * Real Answer :', a)

  print()

  inputs = tokenizer([q]*3, return_tensors="pt").to(model.device)

  torch.manual_seed(42)
  outputs = model.generate(**inputs, max_new_tokens=100)

  for i in range(number_of_answers):
    print('-'*20)
    print(f' * Answer {i+1} : ')
    print(tokenizer.decode(outputs[i], skip_special_tokens=True))
  print('='*50)




 * Question : What's the date of the final of the World Cup 2022?
 * Real Answer : 18-12-2022

--------------------
 * Answer 1 : 
What's the date of the final of the World Cup 2022?

The final of the 2022 World Cup was played on December 18, 2022, at the Lusail Iconic Stadium in Doha, Qatar. Argentina defeated France 4-3 on penalties after a 3-3 draw.
--------------------
 * Answer 2 : 
What's the date of the final of the World Cup 2022?

The final of the 2022 FIFA World Cup was played on December 18, 2022.

--------------------
 * Answer 3 : 
What's the date of the final of the World Cup 2022?

The final of the 2022 FIFA World Cup was played on December 18, 2022, at the Lusail Iconic Stadium in Doha, Qatar.

 * Question : What's the date of la prise de la Bastille?
 * Real Answer : 14-07-1789

--------------------
 * Answer 1 : 
What's the date of la prise de la Bastille?

The date of the taking of the Bastille is July 14, 1789.

---
**Additional Notes**

*   **Historical Context:** 

First, we can see that we don't get only the date, but we also get answers that include additional details that are not useful for our purpose.

Second, All dates are returned in the format "Month Day, Year".

So, the question is : **How can we obtain only the date, and how can we control its format?**

The naive approach is prompt engineering.

In [9]:
questions = [
    f"What's the date of {e}? Only respond with the date and use the format YYYY-MM-DD"
    for e in events
]

number_of_answers = 3

for q, a in zip(questions, answers):
  print(' * Question :', q)
  print(' * Real Answer :', a)

  print()

  inputs = tokenizer([q]*3, return_tensors="pt").to(model.device)

  torch.manual_seed(42)
  outputs = model.generate(**inputs, max_new_tokens=100)

  for i in range(number_of_answers):
    print('-'*20)
    print(f' * Answer {i+1} : ')
    print(tokenizer.decode(outputs[i], skip_special_tokens=True))
  print('='*50)

 * Question : What's the date of the final of the World Cup 2022? Only respond with the date and use the format YYYY-MM-DD
 * Real Answer : 18-12-2022

--------------------
 * Answer 1 : 
What's the date of the final of the World Cup 2022? Only respond with the date and use the format YYYY-MM-DD.

2022-07-09

--------------------
 * Answer 2 : 
What's the date of the final of the World Cup 2022? Only respond with the date and use the format YYYY-MM-DD.

2022-07-09

--------------------
 * Answer 3 : 
What's the date of the final of the World Cup 2022? Only respond with the date and use the format YYYY-MM-DD.

2022-07-09

2022-07-10

2022-07-11

2022-07-12

 * Question : What's the date of la prise de la Bastille? Only respond with the date and use the format YYYY-MM-DD
 * Real Answer : 14-07-1789

--------------------
 * Answer 1 : 
What's the date of la prise de la Bastille? Only respond with the date and use the format YYYY-MM-DD.

21-06-1789
21-07-1789
21-08-1789
21-09-1789
21-10-17

While the model appears to understand the instruction, the results are unreliable: some dates are incorrect, and in some cases the output degenerates into gibberish.

An easier task to consider is information extraction : given a text, extract the information. We will try to extract dates from a text in a specific format (instead of asking the model to retrieve it from its own "memory"). 

In [10]:
events_description = [
    "The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.",

    "La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day.",

    "The independence of El Salvador was declared on 15 September 1821, when Central American provinces broke away from Spanish colonial rule. Influenced by regional independence movements, local leaders signed the Act of Independence in Guatemala City, ending centuries of Spanish control and shaping El Salvador’s national identity."
]


prompts = [
    f"Extract the date of the event from its description : {e}"
    for e in events_description
]

number_of_answers = 3

for p in prompts:
  print(' * Prompt :')
  print(p)
  print()

  inputs = tokenizer([p]*3, return_tensors="pt").to(model.device)

  torch.manual_seed(42)
  outputs = model.generate(**inputs, max_new_tokens=100)

  for i in range(number_of_answers):
    print('-'*20)
    print(f' * Answer {i+1} : ')
    print(tokenizer.decode(outputs[i], skip_special_tokens=True))
  print('='*50)

 * Prompt :
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.

--------------------
 * Answer 1 : 
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals. The event was hosted in Lusail, Qatar.

Here's the date extracted: 2022-12-18

```
Final of the FIFA World Cup 2022
Took place in Lusail, Qatar
Argentina faced France in a dramatic match
Argentina won 4–2 on penalties
Lionel Messi played a decisive role in the match
The event was hosted in Lusail, 

In this setting, the model retrieves the correct information more often. However, it still produces additional text we do not want and the output format is not guaranteed. We can try again prompt engineering. 

In [11]:
events_description = [
    "The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.",

    "La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day.",

    "The independence of El Salvador was declared on 15 September 1821, when Central American provinces broke away from Spanish colonial rule. Influenced by regional independence movements, local leaders signed the Act of Independence in Guatemala City, ending centuries of Spanish control and shaping El Salvador’s national identity."
]


prompts = [
    f"Extract the date of the event from its description : {e}. Return only the date in the format MM-DD-YYYY"
    for e in events_description
]

number_of_answers = 3

for p in prompts:
  print(' * Prompt :')
  print(p)
  print()

  inputs = tokenizer([p]*3, return_tensors="pt").to(model.device)

  torch.manual_seed(42)
  outputs = model.generate(**inputs, max_new_tokens=100)

  for i in range(number_of_answers):
    print('-'*20)
    print(f' * Answer {i+1} : ')
    print(tokenizer.decode(outputs[i], skip_special_tokens=True))
  print('='*50)

 * Prompt :
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.. Return only the date in the format MM-DD-YYYY

--------------------
 * Answer 1 : 
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.. Return only the date in the format MM-DD-YYYY.

Answer: 12-20-2022

--------------------
 * Answer 2 : 
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced

We can see that it works fairly well, but the it fails on some instances. For example in question 2 answer 1 it tries to generate code to extract the date instead of returning the actual date. In the other questions, unwanted text like "Final Answer" is being added to the answer.

**The goal of this section was to illustrate how prompting does not guarantee anything about the output format. We can guide the model, improve the results, but nothing ensures that we will get only what we want.**


In the next section, we implement the solution proposed by Willard & Louf. This approach does not require additional computation resources such as fine-tuning and it provides hard guarantees on the structure of the output.

# Structured outputs

The procedure is fairly simple. First, note that the structure of our output can be written as a regex. Second, we can use that regex to generate an FSA (Finite State Automata) or FSE (Finite State Machine) to guide the sampling process: at each generation step we're in a state of the automata, and the state tells us which tokens we can select to comform to the regex. In other words, at each generation step we select a token that ensures that we get the expected structure of the output. To achieve this, we can set to -inf the logits of the tokens that would break the regex pattern before sampling. Finally, we do the normal sampling process and we update the state our FSA based on the token we selected.  

## Define the regex pattern

In [12]:
import re

# Spaces are represented as _

regex = r"(\d{1,2})_(January|February|March|April|May|June|July|August|September|October|November|December)_(\d{4})"

## Generate the FSM from the regex

In [13]:
!pip install interegular

In [17]:
import interegular

fsm = interegular.parse_pattern(regex).to_fsm()
fsm

fsm(alphabet = Alphabet({'8': 0, '5': 0, '2': 0, '3': 0, '9': 0, '0': 0, '7': 0, '4': 0, '1': 0, '6': 0, 'J': 1, 'S': 2, 'O': 3, 'p': 4, 'h': 5, 'N': 6, 'm': 7, 'a': 8, 'b': 9, 'A': 10, 's': 11, 'c': 12, 'M': 13, 'e': 14, anything_else: 15, 'D': 16, 'r': 17, 'o': 18, 'y': 19, 'v': 20, 'l': 21, 'i': 22, 'g': 23, 'u': 24, 'F': 25, '_': 26, 't': 27, 'n': 28}), states = frozenset({0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76}), initial = 0, finals = frozenset({73}), map = {0: {0: 1}, 1: {0: 2, 26: 3}, 2: {26: 3}, 3: {1: 4, 2: 5, 3: 6, 6: 7, 10: 8, 13: 9, 16: 10, 25: 11}, 4: {8: 12, 24: 13}, 5: {14: 14}, 6: {12: 15}, 7: {18: 16}, 8: {4: 17, 24: 18}, 9: {8: 19}, 10: {14: 20}, 11: {14: 21}, 12: {28: 22}, 13: {21: 23, 28: 24}, 14: {4: 25}, 15: {2

Once we have an FSM that tells us which characters are valid from each state, we build a mapping from FSM states to tokens. This mapping determines which tokens are allowed at each generation step.

In [18]:
from copy import copy

def partial_match(state, token):
    """
    Partially match the token to the DFA starting from `state`.

    We iterate over the token's symbols, and at each symbol we check
    if we there's a valid transition transition to another state using that symbol.

    If there is a symbol without any valid transision, we return None. Otherwise,
    we return a tuple that contains the sequence of traversed states.
    """

    traversed_states = (state,)

    # Iterate over the token's symbols, trying at each step to transition
    # to a new DFA state.

    valid_transitions = copy(fsm.map[state])

    first_symbol, rest_of_token = token[0], token[1:]

    # If there are valid transitions from the current state

    if valid_transitions:

      # For each possible transition
      # see if the beginning of first symbol of token
      # is the start from one (should be all, but here we
      # only do one) of the letters that
      # takes us from `state` to another valid state

      for valid_letter_fsm_id, next_state in valid_transitions.items():

        # Select the characters that are accepted by this transition
        valid_char = [char for char, letter_id in fsm.alphabet.items() if letter_id == valid_letter_fsm_id]



        # Check if one of the characters matches the first symbol
        for char in valid_char:
          if char==first_symbol:

            # If token was only one character long, return
            # `(state, next_state)` because and this is a valid token
            # to get from `state` to `next_state`

            if len(rest_of_token)==0:
              return traversed_states + (next_state,)

            # If token was two characters or more, check that once
            # we have transitioned to `next_state` the remaining symbols
            # in token (contained in `rest_of_token`) constitute a valid
            # sequence of transitions as well

            else:

              next_traversed_states = partial_match(state=next_state, token=rest_of_token)

              # Make sure to inform the recursive parent that
              # this is not an accepted path
              if next_traversed_states is None:
                return None

              # If this is an accepted path
              else:
                return traversed_states + next_traversed_states

      # If we arrived here, this means that `first_symbol` does
      # not allow to make valid transition from `state`

      return None

In [19]:
from collections import defaultdict

def build_state_to_valid_ids_dict(vocabulary:dict[str, int], fsm):

  states_to_ids = defaultdict(dict)

  for state in fsm.states:
    for token, id_ in vocabulary.items():
      # print(f'Token :{token} and id : {id_} and state : {state}')
      # If token represents an invalid transition
      traversed_states = partial_match(state, token)
      if traversed_states:
        states_to_ids[state][id_] = traversed_states[-1]

  return states_to_ids


In [20]:
states_to_valid_ids = build_state_to_valid_ids_dict(tokenizer.vocab, fsm)

## Mask the invalid tokens at each generation step

In [21]:
def generate_structured_outputs(prompt, states_to_valid_ids, model, tokenizer, max_tokens=100):
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    current_fsm_state = 0
    generated_tokens = []

    vocab_size = model.config.vocab_size

    for i in range(max_tokens):
        scores = model(**inputs).logits[:, -1, :]

        # Mask scores based on the current state
        allowed_transitions = states_to_valid_ids[current_fsm_state]
        allowed_token_ids = list(allowed_transitions.keys())

        invalid_ids = [tid for tid in allowed_token_ids if tid < 0 or tid >= vocab_size]
        if invalid_ids:
            raise ValueError(f"Invalid token IDs at state {current_fsm_state}: {invalid_ids[:10]}... (vocab size: {vocab_size})")

        if len(allowed_token_ids) == 0:
            if current_fsm_state in fsm.finals:
                break
            else:
              raise ValueError(f"No allowed tokens at state {current_fsm_state}")

        # Create mask
        mask = torch.ones_like(scores, dtype=torch.bool)
        mask[:, allowed_token_ids] = False
        scores.masked_fill_(mask, -torch.inf)

        # Sample based on the new logits
        probs = torch.softmax(scores, dim=-1)

        torch.manual_seed(42)
        next_token = torch.multinomial(probs[0], num_samples=1).item()

        if next_token == tokenizer.eos_token_id:
            break

        generated_tokens.append(next_token)

        next_token_tensor = torch.tensor([[next_token]], device=model.device)
        inputs = {
            'input_ids': torch.cat([inputs['input_ids'], next_token_tensor], dim=1),
            'attention_mask': torch.cat([inputs['attention_mask'], torch.ones_like(next_token_tensor)], dim=1)
        }

        # Update the current_fsm_state
        current_fsm_state = states_to_valid_ids[current_fsm_state][next_token]

    return tokenizer.decode(generated_tokens, skip_special_tokens=True).replace('_', ' ')

In [22]:
events_description = [
    "The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.",

    "La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day.",

    "The independence of El Salvador was declared on 15 September 1821, when Central American provinces broke away from Spanish colonial rule. Influenced by regional independence movements, local leaders signed the Act of Independence in Guatemala City, ending centuries of Spanish control and shaping El Salvador’s national identity."
]

prompts = [
    f"Extract the date of the event from its description : {e} "
    for e in events_description
]

for p in prompts:


  output = generate_structured_outputs(p, states_to_valid_ids, model, tokenizer)

  print(' * Prompt :')
  print(p)
  print()

  print(' * Output :')
  print(output)
  print('='*50)

 * Prompt :
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals. 

 * Output :
18 December 2022
 * Prompt :
Extract the date of the event from its description : La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day. 

 * Output :
17 October 2023
 * Prompt :
Extract the date of the event from its description : The independence of El Salvador was declared on 15 September 1821, when Central American provinces broke away from Spanish colonial rule. Influenced by regional independence mo

We can see that _all_ the examples follow the desired format. Since at inference time we selected only tokens that generated a valid sequence for our regex, we are 100% certain that the output follows the desired format. This is not the case with prompt enginering. 

Note that the model got the second example wrong. This will be discussed in the next section. 

We can combine structured outputs with prompt engineering to get the right answer AND consistent (and guaranteed) format.

In [23]:
events_description = [
    "The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.",

    "La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day.",

    "The independence of El Salvador was declared on 15 September 1821, when Central American provinces broke away from Spanish colonial rule. Influenced by regional independence movements, local leaders signed the Act of Independence in Guatemala City, ending centuries of Spanish control and shaping El Salvador’s national identity."
]

better_prompts = [
    f"Extract the date of the event from its description : {e} The event mentionned in the text took place on"
    for e in events_description
]

for p in better_prompts:


  output = generate_structured_outputs(p, states_to_valid_ids, model, tokenizer)

  print(' * Prompt :')
  print(p)
  print()

  print(' * Output :')
  print(output)
  print('='*50)

 * Prompt :
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals. The event mentionned in the text took place on

 * Output :
18 December 2022
 * Prompt :
Extract the date of the event from its description : La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day. The event mentionned in the text took place on

 * Output :
14 July 1789
 * Prompt :
Extract the date of the event from its description : The independence of El Salvador was declared on 15 September 1821, when Central America

Once the information is extracted, it can be processed automatically (e.g. passed to an API). Enforcing a new format is also trivial: we simply change the regex, rather than redesigning prompts.

In [24]:
regex = r"(\d{1,2})-(\d{1,2})-(\d{4})" # Format dd-mm-yyyy

fsm = interegular.parse_pattern(regex).to_fsm()

new_states_to_valid_ids = build_state_to_valid_ids_dict(tokenizer.vocab, fsm)

events_description = [
    "The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals.",

    "La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day.",

    "The independence of El Salvador was declared on 15 September 1821, when Central American provinces broke away from Spanish colonial rule. Influenced by regional independence movements, local leaders signed the Act of Independence in Guatemala City, ending centuries of Spanish control and shaping El Salvador’s national identity."
]

better_prompts = [
    f"Extract the date of the event from its description : {e} The event mentionned in the text took place on"
    for e in events_description
]

for p in better_prompts:


  output = generate_structured_outputs(p, new_states_to_valid_ids, model, tokenizer)

  print(' * Prompt :')
  print(p)
  print()

  print(' * Output :')
  print(output)
  print('='*50)

 * Prompt :
Extract the date of the event from its description : The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals. The event mentionned in the text took place on

 * Output :
18-12-2022
 * Prompt :
Extract the date of the event from its description : La prise de la Bastille occurred on 14 July 1789 in Paris, France, and marked a turning point in the French Revolution. Revolutionaries stormed the Bastille prison, a symbol of royal authority and oppression. The event galvanized popular resistance and is commemorated annually as Bastille Day. The event mentionned in the text took place on

 * Output :
14-07-1789
 * Prompt :
Extract the date of the event from its description : The independence of El Salvador was declared on 15 September 1821, when Central American provin

# Limitations of this method

## Distortion of the model’s distribution

A key limitation is that constrained generation does not reflect the model’s true probability distribution.

Consider an LLM with tokens such as 'A', 'ugust', 'pple', 'December', digits (among other tokens) and the task of extracting a date in the format "Month Day, Year".
Because many sentences begin with the letter "A", the token 'A' has a high probability of being selected as the first token. Moreover, 'A' is a valid token under the regex, since it can be followed by 'ugust' or 'pril'.If the model selects 'A' at the first step, it is now constrained to the months of August and April, even if the correct answer is December. As a result, enforcing structure can increase the probability of an incorrect answer. 


This issue is discussed in https://arxiv.org/pdf/2410.13111

In [25]:
regex = r"(January|February|March|April|May|June|July|August|September|October|November|December)_(\d{1,2}),_(\d{4})" # Format month dd, yyyy

fsm = interegular.parse_pattern(regex).to_fsm()

states_to_valid_ids_pb = build_state_to_valid_ids_dict(tokenizer.vocab, fsm)

In [29]:

e = "The FIFA World Cup Final 2022 took place on 18 December 2022 in Lusail, Qatar. Argentina faced France in a dramatic match that ended 3–3 after extra time. Argentina won 4–2 on penalties, with Lionel Messi playing a decisive role in one of football’s most memorable finals."
p = f"Extract the date of the event from its description : {e} Return the date in the format 'month day, year'. The date of the event is"

output = generate_structured_outputs(p, states_to_valid_ids_pb, model, tokenizer)

output

'November 20, 2022'

As we can see even if we use the same prompt as before, by swapping the order of the day and the month we obtain the wrong result. 

## Cost of FSM construction

The second limitation of this method is that generating the FSA from the regex can take a lot of time (although it can generated only once and then be stored for later usage). 

## Expressiveness of regular expressions
A final limitation of this approach is that it only applies to output structures that can be expressed as regular languages. More complex constraints such as generating valid Python code or SQL queries can't be captured by regular expressions in general.